In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

In [ ]:
d = pd.read_csv('raw/Barcelona_2024.csv', parse_dates=['ETAUTC', 'ETDUTC'], decimal=',')
d = d[~d.TERMINALCODI.isna()]
d = d[~d.MOLLMODULS.isna()]
d = d[d.ANYESCALA == 2024]
d = d[d.ESCALAESTAT == 'Finalitzada']
d = d[d.VAIXELLTIPUS == 'Portacontenidors']
d = d[d.MOLLCODI.isin(['36A', '24B'])]
d = d[['TERMINALCODI', 'TERMINALNOM', 'ETAUTC', 'ETDUTC', 'MOLLCODI', 'ESLORA_METRES', 'CALAT_METRES', 'MANEGA_METRES', 'MOLLMODULS', 'ESCALANUM']]
d = d.rename(columns={
    'TERMINALCODI': 'terminal_code',
    'TERMINALNOM': 'terminal_name',
    'ETAUTC': 'eta_utc',
    'ETDUTC': 'etd_utc',
    'MOLLCODI': 'berth_code',
    'ESLORA_METRES': 'ship_length',
    'CALAT_METRES': 'ship_draught',
    'MANEGA_METRES': 'ship_width',
    'MOLLMODULS': 'berth_modulus',
    'ESCALANUM': 'call_num'
})
d['call_time'] = d.etd_utc - d.eta_utc
d = d[d.call_time <= pd.Timedelta(hours=48)]
d[['modulus_start', 'modulus_end']] = d.berth_modulus.str.split('-', expand=True)
d.modulus_start = d.modulus_start.astype(int)
d.modulus_end = d.modulus_end.astype(int)
d['modulus_used'] = d.modulus_end - d.modulus_start + 1
d['month'] = d.eta_utc.dt.month_name()
d['week'] = d.eta_utc.dt.isocalendar().week
d['eta_day_of_year'] = d.eta_utc.dt.day_of_year

In [ ]:
n_berths_tot = {
    '36A': 31,
    '24B': 23
}
n_berths_unused = {
    '36A': 2,
    '24B': 3
}
discretisation_hr = 1

In [ ]:
def generate_quay_info(quay_code):
    data = dict()
    data['n_berths'] = n_berths_tot[quay_code] - n_berths_unused[quay_code]
    data['berth_len'] = [1] * data['n_berths']
    return data

In [ ]:
def generate_instance(quay_code, week_start):
    data = generate_quay_info(quay_code=quay_code)

    th_begin = datetime.strptime(f"2024-W{week_start}-1", "%Y-W%W-%w") # 1 == Monday
    th_end = th_begin + timedelta(days=14)

    df = d[(d.berth_code == quay_code) & (d.eta_utc >= th_begin) & (d.etd_utc <= th_end)].copy()
    df['eta_relative'] = df.eta_utc - th_begin
    df['etd_relative'] = df.etd_utc - th_begin
    df['mod_start_relative'] = df.modulus_start - n_berths_unused[quay_code]
    df['mod_end_relative'] = df.modulus_end - n_berths_unused[quay_code]

    ships = list()
    for _, row in df.iterrows():
        ship = dict()

        eta_rel = row.eta_relative
        if eta_rel.total_seconds() > 7 * 24 * 60 * 60:
            eta_rel -= timedelta(days=7)

        arrival_time = int(np.floor(eta_rel.total_seconds() / 3600 * discretisation_hr))
        handling_time = int(np.floor((row.etd_relative - row.eta_relative).total_seconds() / (3600 * discretisation_hr)))

        ship['id'] = len(ships)
        ship['arrival_time'] = arrival_time
        ship['handling_time'] = handling_time
        ship['length'] = int(row.modulus_used)

        ships.append(ship)

    data['n_ships'] = len(ships)
    data['n_periods'] = int(np.ceil(2 * 7 * 24 / discretisation_hr))
    data['arrival_time'] = [ship['arrival_time'] for ship in ships]
    data['handling_time'] = [ship['handling_time'] for ship in ships]
    data['ship_len'] = [ship['length'] for ship in ships]

    return data

In [ ]:
for quay in ('36A', '24B'):
    for week_start in range(2, 51):
        data = generate_instance(quay_code=quay, week_start=week_start)
        name = f"{quay}_{week_start}.json"

        with open(f"../instances/hyb_dyn_fix_max-comp/bcn_{quay}_{week_start}.json", mode='w') as f:
            json.dump(data, f, indent=2)